In [12]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
from pathlib import Path
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cpu


In [9]:
data_path = Path.cwd().parent / 'data'

In [10]:
product_catalog = pd.read_csv(data_path / 'product_catalog.csv')
product_catalog.head()

,sku,product_name,description
0,PA-BR-1000,AutoPro Brake Pad Set for Ford F-150,Heavy-duty brake pad set designed for Ford F-1...
1,PA-RA-1001,DuraMax Radiator for Honda Civic,Long-life radiator designed for Honda Civic mo...
2,PA-OX-1002,DuraMax Oxygen Sensor for Toyota Camry,High-performance oxygen sensor designed for To...
3,PA-CO-1003,ProStop Control Arm for Ford F-150,Heavy-duty control arm designed for Ford F-150...
4,PA-BR-1004,OEM-Fit Brake Pad Set for Nissan Rogue,Premium brake pad set designed for Nissan Rogu...


In [4]:
product_catalog.shape

(150, 3)

Due to the small size of the product catalog, we can simply just keep all the embeddings in memory during runtime. However, this is not a feasible strategy in practice.

In [8]:
embedding_model = SentenceTransformer(
    'Qwen/Qwen3-Embedding-0.6B',
    device=device,
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 690.90it/s]


In [11]:
# create the embeddings for the product catalog
corpus_embeddings = embedding_model.encode(
    product_catalog['description'].tolist(),
    convert_to_numpy=True,
    device=device,
)

In [25]:
# save the embeddings to quickly reload them at a later date
np.save(data_path / 'embeddings.npy', corpus_embeddings)

In [16]:
# test the pipeline
test_query = 'I need brake pads for a 2019 Toyota Camry that are quiet and long-lasting'

In [22]:
def get_topk_similar(
        embedding_model,
        query,
        corpus_embeddings,
        corpus_labels,
        corpus_skus,
        top_k=3
):
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)

    top_similar = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)[0]

    print(f'Top {top_k} results:')
    for result in top_similar:
        idx = result['corpus_id']
        score = result['score']

        label_match = corpus_labels[idx]
        sku_match = corpus_skus[idx]
        print(f'Score: {score}, Product: {label_match}, SKU: {sku_match}')

    return [corpus_skus[result['corpus_id']] for result in top_similar]

In [20]:
corpus_labels = product_catalog['product_name'].tolist()
corpus_skus = product_catalog['sku'].tolist()

In [24]:
test_similar = get_topk_similar(
    embedding_model=embedding_model,
    query=test_query,
    corpus_embeddings=corpus_embeddings,
    corpus_labels=corpus_labels,
    corpus_skus=corpus_skus,
)
test_similar

Top 3 results:
Score: 0.8278185129165649, Product: ProStop Brake Pad Set for Toyota Camry, SKU: PA-BR-1073
Score: 0.8110172748565674, Product: DuraMax Brake Pad Set for Toyota Camry, SKU: PA-BR-1096
Score: 0.7546431422233582, Product: EnduroGold Brake Pad Set for Chevrolet Silverado, SKU: PA-BR-1067


['PA-BR-1073', 'PA-BR-1096', 'PA-BR-1067']